# FastFlow printer: reproduction strong DeiT

Notebook использует общий pipeline проекта и frozen data-study manifest. По умолчанию он только загружает три сохранённых calibration-запуска strong DeiT и не выполняет inference.

Для ручного воспроизведения выберите один seed и новый `TRY_NUMBER`, затем последовательно используйте `train_calibrate` и, только после заморозки решения, `test`. Текущий test уже раскрыт предыдущими экспериментами, поэтому его результат является ретроспективным.

In [ ]:
import gc
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")

import pandas as pd
import torch
from IPython.display import Image as NotebookImage, display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (cwd, cwd.parent)
    if (candidate / "datasets").is_dir()
    and (candidate / "experiments").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from fastflow_printer_pipeline import (
    PRINTER_CONFIGS,
    file_sha256,
    load_approved_manifest,
    load_experiment_result,
    run_experiment,
)

DATASET_ROOT = PROJECT_ROOT / "datasets" / "processed_printer_dataset_384"
MANIFEST_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "printer"
    / "dataset_v384_audit"
    / "printer_split_v2_data_study.csv"
)
SPLIT_CONFIG_PATH = PROJECT_ROOT / "configs" / "printer_split_v2_data_study.json"
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments" / "printer"


In [ ]:
# load_frozen: read committed calibration artifacts only
# train_calibrate/test: run one explicit seed and try
EXECUTION_MODE = "load_frozen"
CONFIG_NAME = "deit_data_strong_aug_1000"
SEED = 42
TRY_NUMBER = 4
ALLOW_OVERWRITE = False
ALLOW_RETROSPECTIVE_TEST = False
DETERMINISTIC = True
BOOTSTRAP_ITERATIONS = 2000

FROZEN_RUNS = [(1, 42), (2, 123), (3, 2025)]
DISPLAY_SEED = 42
EXPECTED_MANIFEST_SHA256 = (
    "972f3456771e7190aaa8b1101f4de7b19d96ac19a911d004a65fdc62a7909a39"
)

assert EXECUTION_MODE in {"load_frozen", "train_calibrate", "test"}
assert CONFIG_NAME == "deit_data_strong_aug_1000"
if EXECUTION_MODE == "test" and not ALLOW_RETROSPECTIVE_TEST:
    raise RuntimeError("Set ALLOW_RETROSPECTIVE_TEST=True explicitly")
display(pd.DataFrame([PRINTER_CONFIGS[CONFIG_NAME]]))


In [ ]:
manifest, split_config = load_approved_manifest(
    DATASET_ROOT,
    MANIFEST_PATH,
    SPLIT_CONFIG_PATH,
    verify_hashes=True,
)
manifest_sha256 = file_sha256(MANIFEST_PATH)
pipeline_sha256 = file_sha256(PROJECT_ROOT / "code" / "fastflow_printer_pipeline.py")
assert manifest_sha256 == EXPECTED_MANIFEST_SHA256

split_summary = (
    manifest.groupby(["split", "class_name"])
    .agg(
        images=("path", "size"),
        source_images=("source_group", "nunique"),
        objects=("object_group", "nunique"),
    )
    .reset_index()
)
print(f"split_version={split_config['version']}")
print(f"manifest_sha256={manifest_sha256}")
print(f"pipeline_sha256={pipeline_sha256}")
display(split_summary)


In [ ]:
config = PRINTER_CONFIGS[CONFIG_NAME]
if EXECUTION_MODE == "load_frozen":
    results = [
        load_experiment_result(EXPERIMENTS_ROOT, config, try_number, seed)
        for try_number, seed in FROZEN_RUNS
    ]
else:
    result = run_experiment(
        config=config,
        seed=SEED,
        dataset_root=DATASET_ROOT,
        manifest_path=MANIFEST_PATH,
        split_config_path=SPLIT_CONFIG_PATH,
        experiments_root=EXPERIMENTS_ROOT,
        try_number=TRY_NUMBER,
        run_training=EXECUTION_MODE == "train_calibrate",
        run_calibration=EXECUTION_MODE == "train_calibrate",
        run_test=EXECUTION_MODE == "test",
        allow_overwrite=ALLOW_OVERWRITE,
        deterministic=DETERMINISTIC,
        verify_manifest_hashes=True,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    )
    results = [result]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
summary_df = pd.DataFrame(results)

if EXECUTION_MODE != "load_frozen":
    summary_dir = EXPERIMENTS_ROOT / "printer384_v2_data_study_summaries"
    summary_path = summary_dir / (
        f"fastflow_strong_deit_reproduction_{EXECUTION_MODE}_try_{TRY_NUMBER}.csv"
    )
    if summary_path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite summary: {summary_path}")
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_path, index=False)
    print(f"summary={summary_path}")


In [ ]:
metric_columns = [
    "calibration_source_group_balanced_tile_roc_auc",
    "calibration_object_roc_auc",
    "calibration_source_image_roc_auc",
    "calibration_source_group_balanced_tile_balanced_accuracy",
    "calibration_source_group_balanced_tile_fpr",
    "calibration_source_group_balanced_tile_fnr",
]
test_columns = [column.replace("calibration_", "test_") for column in metric_columns]
available_metrics = [column for column in test_columns if column in summary_df]
if not available_metrics:
    available_metrics = [column for column in metric_columns if column in summary_df]
display(
    summary_df[
        [
            "config_name",
            "seed",
            "selected_top_k_pixels",
            "selected_top_k_fraction",
            *available_metrics,
            "log_dir",
        ]
    ]
)

analysis_dir = EXPERIMENTS_ROOT / "printer384_v2_data_study_analysis"
display(pd.read_csv(analysis_dir / "paired_summary_by_config.csv"))
display(pd.read_csv(analysis_dir / "paired_comparisons.csv"))


In [ ]:
display_row = summary_df.loc[summary_df["seed"] == DISPLAY_SEED]
if not display_row.empty:
    run_dir = Path(display_row.iloc[0]["log_dir"])
    for name in (
        "calibration_training_curves.png",
        "calibration_top_k_sweep.png",
        "calibration_score_distribution.png",
    ):
        path = run_dir / name
        if path.is_file():
            display(NotebookImage(filename=str(path)))

dashboard = analysis_dir / "data_study_dashboard.png"
if dashboard.is_file():
    display(NotebookImage(filename=str(dashboard)))
